In [1]:
# install huggingface transformers
!pip install transformers==4.45.2 -q

# verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Foundation model will run on CPU (~5 min instead of ~30s).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.5 MB/s eta 0:00:00
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# load the pre-trained model from huggingface

from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=0
)

print("Foundation model loaded.")

# quick smoke test - feed it one obviously positive and one obviously negative review
test_reviews = [
    "Produto excelente, chegou rápido, estou muito satisfeita!",  # portuguese: "Excellent product, arrived fast, I'm very satisfied!"
    "Horrível, nunca chegou, péssimo vendedor."                    # "Horrible, never arrived, terrible seller."
]

for review in test_reviews:
    result = sentiment(review)[0]
    print(f"\nText: {review}")
    print(f"  Prediction: {result['label']}, confidence: {result['score']:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Foundation model loaded.

Text: Produto excelente, chegou rápido, estou muito satisfeita!
  Prediction: 5 stars, confidence: 0.8358

Text: Horrível, nunca chegou, péssimo vendedor.
  Prediction: 1 star, confidence: 0.9878


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# load the olist tables
import pandas as pd

base_path = "/content/drive/MyDrive/Colab Notebooks/DATA 6545 Data Science and MLOps /Homework 1"

orders = pd.read_csv(f"{base_path}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{base_path}/olist_order_items_dataset.csv")
order_payments = pd.read_csv(f"{base_path}/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(f"{base_path}/olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{base_path}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{base_path}/olist_sellers_dataset.csv")
category_translation = pd.read_csv(f"{base_path}/product_category_name_translation.csv")

print(f"orders: {orders.shape}")
print(f"order_reviews: {order_reviews.shape}")
print(f"order_items: {order_items.shape}")
print(f"products: {products.shape}")

orders: (99441, 8)
order_reviews: (99224, 7)
order_items: (112650, 7)
products: (32951, 9)


In [5]:
print(f"Total reviews: {len(order_reviews)}")
print(f"Reviews with comment_message (any length): {order_reviews['review_comment_message'].notna().sum()}")

# filter to non-empty comments
with_text = order_reviews[order_reviews['review_comment_message'].notna()].copy()
# strip whitespace and drop anything that's effectively empty after stripping
with_text['review_comment_message'] = with_text['review_comment_message'].str.strip()
with_text = with_text[with_text['review_comment_message'].str.len() > 0]

print(f"Reviews with non-empty text after cleaning: {len(with_text)}")

# sample 500 records, random_state=42 for reproducibility (spec requirement)
sample_500 = with_text.sample(n=500, random_state=42).reset_index(drop=True)
print(f"\nSampled: {len(sample_500)} reviews")
print(f"\nActual review_score distribution in the sample:")
print(sample_500['review_score'].value_counts().sort_index())

Total reviews: 99224
Reviews with comment_message (any length): 40977
Reviews with non-empty text after cleaning: 40950

Sampled: 500 reviews

Actual review_score distribution in the sample:
review_score
1     95
2     30
3     57
4     78
5    240
Name: count, dtype: int64


In [6]:
import time

reviews_list = sample_500['review_comment_message'].tolist()

start = time.time()
predictions = sentiment(reviews_list, batch_size=32, truncation=True)
elapsed = time.time() - start

print(f"Foundation model ran on {len(predictions)} reviews in {elapsed:.1f} seconds")
print(f"\nFirst 5 predictions:")
for i in range(5):
    text_preview = reviews_list[i][:60] + "..." if len(reviews_list[i]) > 60 else reviews_list[i]
    print(f"  [{i}] '{text_preview}'")
    print(f"      → {predictions[i]['label']} (confidence: {predictions[i]['score']:.3f})")
    print(f"      Actual star rating: {sample_500.iloc[i]['review_score']}")
    print()

Foundation model ran on 500 reviews in 1.9 seconds

First 5 predictions:
  [0] 'SEM QUEIXAS'
      → 4 stars (confidence: 0.230)
      Actual star rating: 5

  [1] 'Produto chegou conforme descrito e antes do prazo. Cumpre o ...'
      → 4 stars (confidence: 0.503)
      Actual star rating: 4

  [2] 'demora na entrega. e produto pesado demais'
      → 2 stars (confidence: 0.509)
      Actual star rating: 3

  [3] 'Fui muiro bem atendido.'
      → 4 stars (confidence: 0.414)
      Actual star rating: 5

  [4] 'Muito bom'
      → 5 stars (confidence: 0.646)
      Actual star rating: 5



In [7]:
#   4-5 stars -> positive (1)
#   1-3 stars -> negative (0)

# extract the star number from labels like "4 stars" or "1 star"
def parse_stars(label):
    # label looks like "4 stars" or "1 star"
    return int(label.split()[0])

sample_500['foundation_stars'] = [parse_stars(p['label']) for p in predictions]
sample_500['foundation_confidence'] = [p['score'] for p in predictions]
sample_500['foundation_pred_binary'] = (sample_500['foundation_stars'] >= 4).astype(int)

# also compute the ground truth binary label (same rule)
sample_500['actual_binary'] = (sample_500['review_score'] >= 4).astype(int)

print("Class distribution in the 500-record sample:\n")
print(f"Actually positive (review_score >= 4): {sample_500['actual_binary'].sum()} ({sample_500['actual_binary'].mean():.1%})")
print(f"Actually negative (review_score < 4):  {(~sample_500['actual_binary'].astype(bool)).sum()} ({1-sample_500['actual_binary'].mean():.1%})")
print()
print(f"Foundation model predicted positive: {sample_500['foundation_pred_binary'].sum()} ({sample_500['foundation_pred_binary'].mean():.1%})")
print(f"Foundation model predicted negative: {(~sample_500['foundation_pred_binary'].astype(bool)).sum()} ({1-sample_500['foundation_pred_binary'].mean():.1%})")
print()
print("Star-level breakdown (foundation predicted stars vs actual review_score):")
pd.crosstab(sample_500['foundation_stars'], sample_500['review_score'],
            rownames=['Foundation predicted'], colnames=['Actual']).fillna(0).astype(int)

Class distribution in the 500-record sample:

Actually positive (review_score >= 4): 318 (63.6%)
Actually negative (review_score < 4):  182 (36.4%)

Foundation model predicted positive: 273 (54.6%)
Foundation model predicted negative: 227 (45.4%)

Star-level breakdown (foundation predicted stars vs actual review_score):


Actual,1,2,3,4,5
Foundation predicted,,,,,
1,87,18,16,11,23
2,0,8,7,5,5
3,4,1,15,14,13
4,1,1,7,16,35
5,3,2,12,32,164


In [8]:
# compute foundation model's binary metrics on the 500 records
# then retrain hw2 rf on the full olist data and predict on the same 500

import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# -------- foundation model metrics --------

y_true = sample_500['actual_binary'].values
y_pred_foundation = sample_500['foundation_pred_binary'].values

foundation_metrics = {
    'accuracy':  accuracy_score(y_true, y_pred_foundation),
    'precision': precision_score(y_true, y_pred_foundation),
    'recall':    recall_score(y_true, y_pred_foundation),
    'f1':        f1_score(y_true, y_pred_foundation)
}

print("Foundation model metrics on 500 records:")
for k, v in foundation_metrics.items():
    print(f"  {k}: {v:.4f}")

# -------- build features for hw2 rf (same recipe as yesterday) --------

print("\nBuilding hw2-style features from the full data...")

df = orders.copy()
reviews_agg = order_reviews.groupby("order_id", as_index=False)["review_score"].mean()
df = df.merge(reviews_agg, on="order_id", how="left")
df = df[df["review_score"].notna()].copy()
df["is_positive_review"] = (df["review_score"] >= 4).astype(int)

order_items_agg = order_items.groupby("order_id", as_index=False).agg(
    number_of_items=("order_item_id", "count"),
    total_order_value=("price", "sum"),
    avg_product_price=("price", "mean"),
    max_product_price=("price", "max"),
    min_product_price=("price", "min"),
    total_freight_value=("freight_value", "sum"),
    avg_freight_value=("freight_value", "mean"),
    max_freight_value=("freight_value", "max"),
)
df = df.merge(order_items_agg, on="order_id", how="left")

for c in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["delivery_vs_estimated"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days

payment_agg = order_payments.groupby("order_id")["payment_type"].agg(lambda x: x.mode()[0]).reset_index()
df = df.merge(payment_agg, on="order_id", how="left")

items_sellers = order_items[["order_id", "seller_id"]].merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left"
)
seller_state_agg = items_sellers.groupby("order_id")["seller_state"].agg(lambda x: x.mode()[0]).reset_index()
df = df.merge(seller_state_agg, on="order_id", how="left")

def safe_mode(s):
    s = s.dropna()
    return np.nan if s.empty else s.mode().iloc[0]

items_products = order_items[["order_id", "product_id"]].merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
).merge(category_translation, on="product_category_name", how="left")
items_products["product_category"] = items_products["product_category_name_english"].fillna(
    items_products["product_category_name"]
)
product_cat_agg = items_products.groupby("order_id")["product_category"].agg(safe_mode).reset_index()
df = df.merge(product_cat_agg, on="order_id", how="left")

items_photos = order_items[["order_id", "product_id"]].merge(
    products[["product_id", "product_photos_qty"]], on="product_id", how="left"
)
photos_agg = items_photos.groupby("order_id", as_index=False).agg(
    min_product_photos_qty=("product_photos_qty", "min")
)
df = df.merge(photos_agg, on="order_id", how="left")

hw2_features = [
    "delivery_days", "delivery_vs_estimated", "number_of_items",
    "total_order_value", "avg_product_price", "max_product_price",
    "min_product_price", "total_freight_value", "avg_freight_value",
    "max_freight_value", "product_category", "seller_state",
    "payment_type", "min_product_photos_qty"
]

# -------- train hw2 rf on the full training set --------

print("Training HW2 RF (max_depth=30, n_estimators=100)...")

y = df["is_positive_review"]
X = df[hw2_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

numeric = ["delivery_days", "delivery_vs_estimated", "number_of_items",
           "total_order_value", "avg_product_price", "max_product_price",
           "min_product_price", "total_freight_value", "avg_freight_value",
           "max_freight_value", "min_product_photos_qty"]
categorical = ["product_category", "seller_state", "payment_type"]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])

hw2_rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, max_depth=30, random_state=42, n_jobs=-1)),
])

import time
start = time.time()
hw2_rf.fit(X_train, y_train)
print(f"  Trained in {time.time()-start:.1f}s")

# -------- predict on the same 500 records --------

sample_orders_features = df[df['order_id'].isin(sample_500['order_id'])][['order_id'] + hw2_features]

aligned = sample_500[['order_id', 'actual_binary']].merge(sample_orders_features, on='order_id', how='left')

print(f"\n500 sampled order_ids matched in feature data: {aligned.shape[0]}/500")

y_pred_hw2 = hw2_rf.predict(aligned[hw2_features])

hw2_metrics = {
    'accuracy':  accuracy_score(aligned['actual_binary'], y_pred_hw2),
    'precision': precision_score(aligned['actual_binary'], y_pred_hw2),
    'recall':    recall_score(aligned['actual_binary'], y_pred_hw2),
    'f1':        f1_score(aligned['actual_binary'], y_pred_hw2)
}

print("\nHW2 RF metrics on same 500 records:")
for k, v in hw2_metrics.items():
    print(f"  {k}: {v:.4f}")

Foundation model metrics on 500 records:
  accuracy: 0.8060
  precision: 0.9048
  recall: 0.7767
  f1: 0.8359

Building hw2-style features from the full data...
Training HW2 RF (max_depth=30, n_estimators=100)...
  Trained in 102.4s

500 sampled order_ids matched in feature data: 500/500

HW2 RF metrics on same 500 records:
  accuracy: 0.8360
  precision: 0.8010
  recall: 0.9874
  f1: 0.8845


In [9]:
# 4 spec-required metrics plus negative-class recall and macro f1

from sklearn.metrics import recall_score, f1_score, precision_score

y_true = sample_500['actual_binary'].values
y_pred_fnd = sample_500['foundation_pred_binary'].values

def compute_all(y_true, y_pred):
    return {
        'accuracy':      accuracy_score(y_true, y_pred),
        'precision_pos': precision_score(y_true, y_pred, pos_label=1),
        'recall_pos':    recall_score(y_true, y_pred, pos_label=1),
        'f1_pos':        f1_score(y_true, y_pred, pos_label=1),
        'recall_neg':    recall_score(y_true, y_pred, pos_label=0),
        'f1_macro':      f1_score(y_true, y_pred, average='macro'),
    }

fnd = compute_all(y_true, y_pred_fnd)
hw2 = compute_all(y_true, y_pred_hw2)

comparison = pd.DataFrame({
    'Foundation Model (review text)': [
        fnd['accuracy'], fnd['precision_pos'], fnd['recall_pos'],
        fnd['f1_pos'], fnd['recall_neg'], fnd['f1_macro']
    ],
    'HW2 Model (order features)': [
        hw2['accuracy'], hw2['precision_pos'], hw2['recall_pos'],
        hw2['f1_pos'], hw2['recall_neg'], hw2['f1_macro']
    ]
}, index=[
    'Accuracy',
    'Precision (positive class)',
    'Recall (positive class)',
    'F1 (positive class)',
    'Recall (negative class)',
    'Macro F1'
]).round(4)

print("Part 1B: Model Comparison on Same 500 Records\n")
print(comparison.to_string())

print("\nWinner by metric:")
for metric in comparison.index:
    f = comparison.loc[metric, 'Foundation Model (review text)']
    h = comparison.loc[metric, 'HW2 Model (order features)']
    winner = 'Foundation' if f > h else 'HW2 RF' if h > f else 'Tie'
    print(f"  {metric}: {winner} ({f:.4f} vs {h:.4f})")

Part 1B: Model Comparison on Same 500 Records

                            Foundation Model (review text)  HW2 Model (order features)
Accuracy                                            0.8060                      0.8360
Precision (positive class)                          0.9048                      0.8010
Recall (positive class)                             0.7767                      0.9874
F1 (positive class)                                 0.8359                      0.8845
Recall (negative class)                             0.8571                      0.5714
Macro F1                                            0.7994                      0.8009

Winner by metric:
  Accuracy: HW2 RF (0.8060 vs 0.8360)
  Precision (positive class): Foundation (0.9048 vs 0.8010)
  Recall (positive class): HW2 RF (0.7767 vs 0.9874)
  F1 (positive class): HW2 RF (0.8359 vs 0.8845)
  Recall (negative class): Foundation (0.8571 vs 0.5714)
  Macro F1: HW2 RF (0.7994 vs 0.8009)


## Reflection ##

Looking at this from a business angle, the foundation model performed better. It caught 85.7% of actual negative reviews vs HW2 RF's 57.1%. From the cost-sensitive analysis I did in HW3, I know that missing an unhappy customer is way more expensive than falsely flagging a happy one, so catching negatives is what actually matters. HW2 RF's higher accuracy is mostly an illusion from predicting 'positive' on almost everything (recall on positives was 98.7%).

That said, the foundation model's strong performance came with no Olist-specific training, which has both advantages and disadvantages. The advantages are that it saves time and compute on training since someone already did it on a huge amount of data, it handles Portuguese and other languages without needing translation, and it catches patterns we wouldn't think to engineer as features. The disadvantages are that deployment is expensive since the foundation model is about 670 MB vs our HW2 RF pipeline at 205 KB, and it misses Olist specific nuances. For example, about 10% of actual 5 star reviews got predicted as 1 star, likely because of sarcasm or short reviews where the text doesn't match the rating.

Those tradeoffs shape when each approach makes sense. A foundation model fits when there is unlabeled data, the task is general, or deep learning is needed on text, images, or audio. A custom model fits when there is labeled data specific to your industry, the task is niche, there are limited resources, or you need to explain why the model predicted what it did. Currently there is no pretrained model that predicts review scores from order features, so only a custom model can flag dissatisfied customers before they write a review.

Which is exactly why these two approaches work best combined rather than chosen between. Use the custom HW2 RF before a review is written to predict satisfaction from order features, so customer service can reach out proactively to high risk orders. Then use the foundation model after the review is written to catch anything missed and route negative reviews to a human for response.